# Code-Mapper demo
A THIN notebook: all logic lives in the `codemapper/` package. Run cells top to bottom.
Before committing: **Edit > Clear all outputs** (your old notebooks were carrying ~200 KB of printed output).

In [ ]:
# Colab only: uncomment
# !pip install -q networkx google-genai
# !git clone https://github.com/<you>/<this-repo>.git && %cd <this-repo>
# !git clone --depth 1 https://github.com/mailpile/Mailpile.git target_repo
import sys; sys.path.insert(0, '..' if 'notebooks' in __import__('os').getcwd() else '.')
REPO = 'target_repo'   # <- the repo you want to analyse

In [ ]:
from codemapper import Index, Resolver, build_call_graph, resolution_report, top_blast_radius, trusted
import json

ix = Index.build(REPO)              # parse every file (old Python is repaired and reported, never hidden)
res = Resolver(ix)                  # decide what each call refers to, with a confidence
G = build_call_graph(ix, res)       # networkx graph; every edge has a confidence
print(json.dumps(resolution_report(ix, res), indent=1, default=list))

In [ ]:
# Which functions would hurt most if changed?  (production code only, ambiguous edges hidden)
for fid, score in top_blast_radius(G, 10):
    print(f'{score:5d}  {ix.label(fid)}')

In [ ]:
# Who calls X?   Use Class.method to be specific. ix.find() shows every match.
name = 'gettext'
fid = ix.find(name)[0]
for caller in list(trusted(G).predecessors(fid))[:15]:
    print(ix.label(caller), G[caller][fid]['confidence'])

In [ ]:
# Ask the LLM (needs GEMINI_API_KEY; pin an explicit model name, not '...-latest')
# from codemapper import llm
# ask = llm.make_gemini('PUT-A-PINNED-MODEL-NAME-HERE')
# print(llm.ask(ask, ix, G, fid, 'What would break if this function raised an exception?'))